In [0]:
# imports
from pyspark.sql.types import *
import pyspark.sql.functions as F

In [0]:
# bronze class
class BronzeIngestion:
    def __init__(self, source_file_path, source_file_type, source_file_delimeter, target_catalog, target_schema, target_table_name, target_table_type, target_table_mode):
        self.source_file_path = source_file_path
        self.source_file_type = source_file_type
        self.source_file_delimeter = source_file_delimeter
        self.target_catalog = target_catalog
        self.target_schema = target_schema
        self.target_table_name = target_table_name
        self.target_table_type = target_table_type
        self.target_table_mode = target_table_mode
        self.schema_declaration()

    def schema_declaration(self):
        self.schema = StructType([
            StructField("transaction_id", StringType(), True),
            StructField("transaction_date", DateType(), True),
            StructField("transaction_year", IntegerType(), True),
            StructField("transaction_quarter", IntegerType(), True),
            StructField("transaction_month", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("product_category", StringType(), True),
            StructField("quantity_unit", StringType(), True),
            StructField("supplier_name", StringType(), True),
            StructField("supplier_country", StringType(), True),
            StructField("supplier_reliability_score", DoubleType(), True),
            StructField("refinery_name", StringType(), True),
            StructField("destination_city", StringType(), True),
            StructField("transportation_mode", StringType(), True),
            StructField("ordered_quantity", DoubleType(), True),
            StructField("demand_quantity", DoubleType(), True),
            StructField("available_inventory", DoubleType(), True),
            StructField("unit_price_usd", DoubleType(), True),
            StructField("product_cost_usd", DoubleType(), True),
            StructField("transportation_cost_usd", DoubleType(), True),
            StructField("total_cost_usd", DoubleType(), True),
            StructField("expected_lead_time_days", IntegerType(), True),
            StructField("actual_lead_time_days", IntegerType(), True),
            StructField("delay_days", IntegerType(), True),
            StructField("is_delayed", IntegerType(), True),
            StructField("is_stockout", IntegerType(), True),
            StructField("quality_status", StringType(), True),
            StructField("quality_score", DoubleType(), True),
            StructField("disruption_type", StringType(), True),
            StructField("delivery_status", StringType(), True),
            StructField("ingestion_timestamp", TimestampType(), True),
            StructField("source_system", StringType(), True),
            StructField("operation_type", StringType(), True),
        ])

    def load_data(self):
        print("01: Data Loading Started.")
        self.data = spark.read\
                         .format(self.source_file_type)\
                         .option("header", True)\
                         .option("delimiter", self.source_file_delimeter)\
                         .schema(self.schema)\
                         .load(self.source_file_path)
                        
        print("02: Columns Dropped: [ingestion_timestamp, source_system, operation_type]")
        self.data = self.data.drop("ingestion_timestamp", "source_system", "operation_type")

        print("03: Timestamp Column Added")
        self.data = self.data.withColumn(
            "bronze_ingestion_timestamp",
            F.current_timestamp()
        )

        #self.data.show()
        print("04: Data Loaded in the Dataframe.")

    def write_to_table(self):
        target_path = self.target_catalog + '.' + self.target_schema + '.' + self.target_table_name
        print(f"05: Data Writing started, Path: {target_path}.")

        self.data.write.format(self.target_table_type)\
                       .mode(self.target_table_mode)\
                       .saveAsTable(target_path)

        print("06: Data Writing Done.")

In [0]:
# main part
bronze_ingestion = BronzeIngestion(
    source_file_path="/Volumes/supply_chain/raw/data/supply_chain_dataset.csv",
    source_file_type="csv",
    source_file_delimeter=",",
    target_catalog="supply_chain",
    target_schema="bronze",
    target_table_name="bronze_supply_chain",
    target_table_type="delta",
    target_table_mode="overwrite"
)

bronze_ingestion.load_data()

bronze_ingestion.write_to_table()

In [0]:
%sql
SELECT *
FROM supply_chain.bronze.bronze_supply_chain;

In [0]:
%sql
SELECT
    COUNT(*) AS rows_count
FROM supply_chain.bronze.bronze_supply_chain;